1. Setup

In [1]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.6 MB/s eta 0:00:00


In [2]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [3]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

2. Define  Experts

In [4]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": "You are a strict technical expert. Provide precise, code-focused solutions with minimal explanation."
    },
    "billing": {
        "system_prompt": "You are a polite and empathetic billing support assistant. Help users with payments, refunds, and subscriptions."
    },
    "general": {
        "system_prompt": "You are a friendly general assistant. Answer casually and helpfully."
    }
}

3. The Router (The Core Task)

In [8]:
def route_prompt(user_input):
    prompt = f"""
Classify this text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()
    return category

4. The Orchestrator


In [9]:
def process_request(user_input):
    category = route_prompt(user_input)

    # fallback safety
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return {
        "category": category,
        "response": response.choices[0].message.content
    }

In [10]:
queries = [
    "My python script gives IndexError",
    "I was charged twice for my subscription",
    "Hey how are you?"
]

for q in queries:
    result = process_request(q)
    print("\nQuery:", q)
    print("Category:", result["category"])
    print("Response:", result["response"])


Query: My python script gives IndexError
Category: technical
Response: ```python
try:
    # Your code here
except IndexError as e:
    print(f"Error: {e}")
```
Alternatively, validate your list or tuple index before accessing it:
```python
my_list = [1, 2, 3]
index = 3

if index < len(my_list):
    print(my_list[index])
else:
    print("Index out of range")
```

Query: I was charged twice for my subscription
Category: billing
Response: I'm so sorry to hear that you were charged twice for your subscription. That can be frustrating and confusing. I'm here to help you resolve this issue as quickly as possible.

Can you please provide me with some more information about the duplicate charge? This will help me to better understand the situation and assist you more effectively. Could you please tell me:

1. The date of the duplicate charge
2. The amount of the duplicate charge
3. Your subscription plan and the normal charge amount
4. Have you noticed any other unusual transactions on your a

BONUS

In [11]:
def get_bitcoin_price():
    return "Bitcoin price is $65,000 (mock data)"

In [12]:
def route_prompt(user_input):
    prompt = f"""
Classify this text into one of these categories:
[technical, billing, general, tool]

Return ONLY the category name.

Text: "{user_input}"
"""

In [15]:
def process_request(user_input):
    category = route_prompt(user_input)

    if category == "tool":
        return {
            "category": "tool",
            "response": get_bitcoin_price()
        }

    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return {
        "category": category,
        "response": response.choices[0].message.content
    }

In [16]:
print(process_request("What is the current price of Bitcoin?"))

{'category': 'general', 'response': "I'm a large language model, I don't have real-time access to current market prices. But I can suggest some ways for you to find the current price of Bitcoin.\n\nYou can check the current price of Bitcoin on various cryptocurrency websites, such as CoinMarketCap, CoinGecko, or CryptoCompare. These websites provide up-to-date prices, charts, and other market data.\n\nAlternatively, you can also check the prices on cryptocurrency exchanges like Coinbase, Binance, or Kraken. Keep in mind that prices may vary slightly depending on the exchange and the time of day.\n\nIf you're looking for a quick answer, I can suggest checking a reliable online source, such as a financial news website or a cryptocurrency portal. Just remember that prices can fluctuate rapidly, so it's always a good idea to verify the information through multiple sources."}
